# 02 - Temporal Preprocessing & Target Engineering

**Project:** E-Commerce Dynamic Pricing & Demand Forecasting Engine (Olist Brazil)
**Stage:** Stage 2 | **Status:** executed
**Source modules:** `src/data/*`, `src/features/*`, `src/utils/*`
**Certified by:** `tests/test_data_processing.py`, `tests/test_target_engineering.py`,
`tests/test_temporal_split.py`, `tests/test_metrics.py`

This notebook executes the production Phase-2 pipeline on the real data:

1. **Valid-order fact table** (status policy, order-level economics).
2. **Weekly demand target** `order_demand` per product (dense zero-filled matrix).
3. **Past-only feature stack** (calendar, lags, rolling, price drivers).
4. **Chronological split** train/val/test with hard no-leakage guard.
5. **Conditional imputation** fit strictly on train.
6. **Zero-lookahead probe** and metric-suite validation.


## 1. Setup & Imports

In [1]:
import sys
from pathlib import Path

# Ensure the project root is importable when run from the notebooks directory.
ROOT = Path.cwd().parents[1] if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np

from src.utils.logging_setup import get_logger
from src.utils.metrics import describe_metrics
from src.data.data_processing import build_fact_table
from src.data.target_engineering import (
    build_weekly_demand,
    build_demand_matrix,
    save_demand_matrix,
)
from src.data.temporal_split import temporal_split, assert_chronological_order, save_splits
from src.data.imputation import impute_splits
from src.features.feature_engineering import build_feature_matrix

LOGGER = get_logger("notebook_02")
RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"
SPLITS_DIR = ROOT / "data" / "splits"
LOGGER.info("Project root: %s", ROOT)

2026-09-21 22:11:48 | INFO    | notebook_02 | Project root: D:\ALL Projects\E-Commerce


## 2. Valid-Order Fact Table (Status Policy)

Business rule (documented in `src/data/data_processing.py`): only `canceled`
and `unavailable` orders are removed. Genuinely-placed in-transit orders
(`shipped`, `invoiced`, `processing`, `created`, `approved`) remain - they are
real demand. Filtering to `delivered`-only would truncate the last weeks of the
timeline (delivery latency) and bias recent demand low.

In [2]:
fact = build_fact_table(RAW_DIR)
print(f"Valid-order fact table: {fact.shape}")
print(f"Order purchase window : {fact['order_purchase_timestamp'].min()} -> {fact['order_purchase_timestamp'].max()}")
print("Order status distribution:")
print(fact["order_status"].value_counts().to_string())

2026-09-21 22:11:51 | INFO    | src.data.data_processing | Loaded orders=99441 items=112650 products=32951 translation=71


2026-09-21 22:11:51 | INFO    | src.data.data_processing | merge_core_tables: items=112650, orders=99441


2026-09-21 22:11:52 | INFO    | src.data.data_processing |   +orders  -> 112650 x 14


2026-09-21 22:11:52 | INFO    | src.data.data_processing |   +products -> 112650 x 22


2026-09-21 22:11:52 | INFO    | src.data.data_processing |   +translation -> 112650 x 23


2026-09-21 22:11:52 | INFO    | src.data.data_processing | apply_status_policy: dropped 549 invalid rows (['canceled', 'unavailable']) -> 112101 rows remain


Valid-order fact table: (112101, 23)
Order purchase window : 2016-09-04 21:15:19 -> 2018-09-03 09:06:57
Order status distribution:
order_status
delivered     110197
shipped         1185
invoiced         359
processing       357
approved           3


## 3. Weekly Demand Target (`order_demand`)

Each item row counts as one unit sold. `order_demand` = row count per
(product, week). Weeks use pandas weekly periods (``W``, ending Sunday).

In [3]:
demand_sparse = build_weekly_demand(fact)
print(demand_sparse.shape)
print(demand_sparse.head().to_string())

2026-09-21 22:11:53 | INFO    | src.data.target_engineering | build_weekly_demand: 78940 product-week rows (0.00 of all possible)


(78940, 6)
                         product_id                   week  order_demand  avg_price  avg_freight  product_category_name
0  00066f42aeeb9f3007548bb9d3f33c38  2018-05-14/2018-05-20             1     101.65        18.59             perfumaria
1  00088930e925c41fd95ebfe695fd2655  2017-12-11/2017-12-17             1     129.90        13.93             automotivo
2  0009406fd7479715e4bef61dd91f2462  2017-12-18/2017-12-24             1     229.00        13.10        cama_mesa_banho
3  000b8f95fcb9e0096488278317764d19  2018-07-30/2018-08-05             1      58.90        19.60  utilidades_domesticas
4  000b8f95fcb9e0096488278317764d19  2018-08-06/2018-08-12             1      58.90        19.60  utilidades_domesticas


## 4. Dense Product x Week Matrix

Every (product, week) combination between the first and last observed week is
materialized, zero-filled for demand. A dense matrix is the correct forecasting
artifact - a week with no sales is true ``0`` demand, not a missing record.
Sale-free weeks carry `NaN` price/freight (resolved by train-only imputation).

In [4]:
matrix = build_demand_matrix(demand_sparse)
print(f"Dense matrix: {matrix.shape}")
print(matrix.head().to_string())
print("\nColumn dtypes:\n", matrix.dtypes.to_string())

2026-09-21 22:11:55 | INFO    | src.data.target_engineering | build_demand_matrix: 32729 products x 106 weeks = 3469274 rows


Dense matrix: (3469274, 6)
                         product_id                   week  order_demand  avg_price  avg_freight product_category_name
0  00066f42aeeb9f3007548bb9d3f33c38  2016-08-29/2016-09-04             0        NaN          NaN                   NaN
1  00066f42aeeb9f3007548bb9d3f33c38  2016-09-05/2016-09-11             0        NaN          NaN                   NaN
2  00066f42aeeb9f3007548bb9d3f33c38  2016-09-12/2016-09-18             0        NaN          NaN                   NaN
3  00066f42aeeb9f3007548bb9d3f33c38  2016-09-19/2016-09-25             0        NaN          NaN                   NaN
4  00066f42aeeb9f3007548bb9d3f33c38  2016-09-26/2016-10-02             0        NaN          NaN                   NaN

Column dtypes:
 product_id                      object
week                     period[W-SUN]
order_demand                     int64
avg_price                      float64
avg_freight                    float64
product_category_name         category


## 5. Past-Only Feature Engineering

All features are strictly backward-looking by construction (see
`src/features/feature_engineering.py`): lags via ``shift(k)``, rolling
statistics computed on the shift-by-1 series (window covers ``[t-4, t-1]``),
and price/freight/category-share drivers lagged one week. Calendar features are
stateless derivations of the week label.

In [5]:
features = build_feature_matrix(matrix)
print(f"Feature frame: {features.shape}")
print("Feature columns:")
for c in features.columns:
    print(f"  - {c}")

2026-09-21 22:11:55 | INFO    | src.features.feature_engineering | build_feature_matrix: starting feature construction (3469274 rows).


2026-09-21 22:12:08 | INFO    | src.features.feature_engineering | add_demand_drivers: price_prev, price_chg_1w, freight_prev, category_share_prev added.


2026-09-21 22:12:15 | INFO    | src.features.feature_engineering | add_lag_features: lags (1, 2, 3, 4) added.


2026-09-21 22:12:33 | INFO    | src.features.feature_engineering | add_rolling_features: windows (4,) added (shifted-by-1, past-only).


2026-09-21 22:12:34 | INFO    | src.features.feature_engineering | build_feature_matrix: completed -> 3469274 x 21


Feature frame: (3469274, 21)
Feature columns:
  - product_id
  - week
  - order_demand
  - avg_price
  - avg_freight
  - product_category_name
  - week_of_year
  - month
  - quarter
  - year
  - price_prev
  - freight_prev
  - price_chg_1w
  - category_share_prev
  - demand_lag_1
  - demand_lag_2
  - demand_lag_3
  - demand_lag_4
  - rolling_demand_mean_4w
  - rolling_demand_max_4w
  - demand_log1p


## 6. Chronological Split (No-Shuffle, No-Leakage Boundary)

The feature frame is partitioned purely by time. `assert_chronological_order`
is a hard trip-wire: if any training observation ever out-times a validation
one, the pipeline aborts.

In [6]:
splits = temporal_split(features, "week", ratios=(0.70, 0.15, 0.15))
assert_chronological_order(splits, "week")
for name, frame in splits.items():
    span = (frame["week"].max() - frame["week"].min()).n
    LOGGER.info("  %-5s rows=%6d weeks=%3d (n_weeks span %d)", name, len(frame), frame["week"].nunique(), span)

w_train = splits["train"]["week"].max()
w_val = splits["val"]["week"].min()
w_test = splits["test"]["week"].min()
print(f"Boundaries: train <= {w_train} | val {w_val}.. | test >= {w_test}")
# A weekly-period boundary may be cut mid-week (rows within the same week are
# still strictly <= the boundary), so <= is the correct ordering guard here.
assert w_train <= w_val <= w_test, "Hard chronology violation in split boundaries."
assert splits["train"]["week"].max() < splits["test"]["week"].min(), "train must be strictly before test"

2026-09-21 22:12:42 | INFO    | src.data.temporal_split | temporal_split: train=2428492 val=520391 test=520391 (total=3469274)


2026-09-21 22:12:42 | INFO    | notebook_02 |   train rows=2428492 weeks= 75 (n_weeks span 74)


2026-09-21 22:12:42 | INFO    | notebook_02 |   val   rows=520391 weeks= 17 (n_weeks span 16)


2026-09-21 22:12:42 | INFO    | notebook_02 |   test  rows=520391 weeks= 16 (n_weeks span 15)


Boundaries: train <= 2018-01-29/2018-02-04 | val 2018-01-29/2018-02-04.. | test >= 2018-05-21/2018-05-27


## 7. Conditional Imputation (Fit on Train Only)

Statistical statistics (median) are learned **only from the training split**.
Categorical unknowns are remapped to the reserved label `unknown`. Demand-based
features were already zero-filled (semantically "no prior demand"); only the
price/freight driver columns still carry NaNs and enter imputation.

In [7]:
CATEGORICAL_COLS = ["product_category_name"]
NUMERIC_COLS = ["avg_price", "avg_freight", "price_prev", "price_chg_1w", "freight_prev"]

splits_imputed = impute_splits(splits, categorical_cols=CATEGORICAL_COLS, numeric_cols=NUMERIC_COLS)

# Post-condition: every split must be NaN-free on all modeling columns.
for name in splits_imputed:
    remaining = int(splits_imputed[name].isna().sum().sum())
    assert remaining == 0, f"{name} still has {remaining} NaNs"
    print(f"{name}: NaN-free ({splits_imputed[name].shape})")

2026-09-21 22:12:52 | INFO    | src.data.imputation | impute_splits: test remaining NaNs=0


2026-09-21 22:12:53 | INFO    | src.data.imputation | impute_splits: train remaining NaNs=0


2026-09-21 22:12:53 | INFO    | src.data.imputation | impute_splits: val remaining NaNs=0


train: NaN-free ((2428492, 21))


val: NaN-free ((520391, 21))
test: NaN-free ((520391, 21))


## 8. Zero-Lookahead Leakage Probe

Behavioral proof that no engineered feature at week `t` depends on demand,
price, or freight observed **at** week `t`. We nullify a product-week's current
facts and rebuild the stack; features *at that same week* must be bit-identical.
The probe runs on the 6 highest-volume products to stay fast; the identical
method is certified against the full upstream pipeline in the test suite.

In [8]:
ENGINEERED_COLS = [
    "week_of_year", "month", "quarter", "year",
    "demand_lag_1", "demand_lag_2", "demand_lag_3", "demand_lag_4",
    "rolling_demand_mean_4w", "rolling_demand_max_4w",
    "price_prev", "price_chg_1w", "freight_prev", "category_share_prev",
]

top_products = (
    demand_sparse.groupby("product_id")["order_demand"].sum().nlargest(6).index.tolist()
)
small_matrix = build_demand_matrix(demand_sparse[demand_sparse["product_id"].isin(top_products)])
baseline = build_feature_matrix(small_matrix)

for prod in top_products:
    rows = baseline[baseline["product_id"] == prod]
    mid_week = rows.iloc[len(rows) // 2]["week"]
    perturbed = small_matrix.copy()
    mask = (perturbed["product_id"] == prod) & (perturbed["week"] == mid_week)
    perturbed.loc[mask, ["order_demand", "avg_price", "avg_freight"]] = np.nan
    rebuilt = build_feature_matrix(perturbed)

    orig = baseline[mask.reindex(baseline.index).fillna(False)][ENGINEERED_COLS].to_numpy(dtype=float)
    new = rebuilt[mask.reindex(rebuilt.index).fillna(False)][ENGINEERED_COLS].to_numpy(dtype=float)
    assert np.allclose(orig, new, equal_nan=True), f"LEAK detected for {prod} at {mid_week}"
    LOGGER.info("probe OK: %s @ %s features invariant under current-week nullification", prod, mid_week)

print("No-lookahead probe passed for all sampled product-weeks.")

2026-09-21 22:12:54 | INFO    | src.data.target_engineering | build_demand_matrix: 6 products x 78 weeks = 468 rows


2026-09-21 22:12:54 | INFO    | src.features.feature_engineering | build_feature_matrix: starting feature construction (468 rows).


2026-09-21 22:12:54 | INFO    | src.features.feature_engineering | add_demand_drivers: price_prev, price_chg_1w, freight_prev, category_share_prev added.


2026-09-21 22:12:54 | INFO    | src.features.feature_engineering | add_lag_features: lags (1, 2, 3, 4) added.


2026-09-21 22:12:54 | INFO    | src.features.feature_engineering | add_rolling_features: windows (4,) added (shifted-by-1, past-only).


...

2026-09-21 22:12:55 | INFO    | src.features.feature_engineering | add_lag_features: lags (1, 2, 3, 4) added.


2026-09-21 22:12:55 | INFO    | src.features.feature_engineering | add_rolling_features: windows (4,) added (shifted-by-1, past-only).


2026-09-21 22:12:55 | INFO    | src.features.feature_engineering | build_feature_matrix: completed -> 468 x 21


2026-09-21 22:12:55 | INFO    | notebook_02 | probe OK: 53759a2ecddad2bb87a079a1f1519f73 @ 2017-11-27/2017-12-03 features invariant under current-week nullification


No-lookahead probe passed for all sampled product-weeks.


## 9. Persist Artifacts

* `data/processed/demand_matrix_weekly.csv.gz` - dense target/driver matrix.
* `data/splits/{train,val,test}.csv` - imputed, feature-engineered splits.

Raw files remain untouched (ingestion-only, NTFS read-only).

In [9]:
matrix_path = save_demand_matrix(matrix, PROCESSED_DIR / "demand_matrix_weekly.csv.gz")
write_splits = save_splits(splits_imputed, SPLITS_DIR)
for name, path in write_splits.items():
    print(f"  {name:<5} -> {path}")

2026-09-21 22:13:51 | INFO    | src.data.target_engineering | save_demand_matrix: wrote D:\ALL Projects\E-Commerce\data\processed\demand_matrix_weekly.csv.gz (13.7 MB)


2026-09-21 22:15:05 | INFO    | src.data.temporal_split | save_splits: wrote D:\ALL Projects\E-Commerce\data\splits\train.csv (2428492 rows)


2026-09-21 22:15:21 | INFO    | src.data.temporal_split | save_splits: wrote D:\ALL Projects\E-Commerce\data\splits\val.csv (520391 rows)


2026-09-21 22:15:36 | INFO    | src.data.temporal_split | save_splits: wrote D:\ALL Projects\E-Commerce\data\splits\test.csv (520391 rows)


  train -> D:\ALL Projects\E-Commerce\data\splits\train.csv
  val   -> D:\ALL Projects\E-Commerce\data\splits\val.csv
  test  -> D:\ALL Projects\E-Commerce\data\splits\test.csv


## 10. Metric Suite Validation (Scaffold)

The Phase-2 metric scaffold is exercised on a no-skill baseline (`predict = mean
of train`) and a perfect oracle to demonstrate the exact WAPE / MAE / RMSE /
Asymmetric-Loss semantics the Phase-3 models will be measured against.

In [10]:
train_target = splits_imputed["train"]["order_demand"].to_numpy()
val_target = splits_imputed["val"]["order_demand"].to_numpy()
pred_mean = np.full_like(val_target, train_target.mean(), dtype=float)
pred_oracle = val_target.copy()

mean_baseline = describe_metrics(val_target, pred_mean)
oracle = describe_metrics(val_target, pred_oracle)
print("Baseline (train-mean):")
for k, v in mean_baseline.items():
    print(f"  {k:<20} {v:.4f}")
print("Oracle (perfect):")
for k, v in oracle.items():
    print(f"  {k:<20} {v:.4f}")
assert oracle["wape"] == 0.0 and oracle["mae"] == 0.0 and oracle["rmse"] == 0.0

Baseline (train-mean):
  wape                 1.3783
  mae                  0.0810
  rmse                 0.4365
  asymmetric_loss_2_1  0.1388
Oracle (perfect):
  wape                 0.0000
  mae                  0.0000
  rmse                 0.0000
  asymmetric_loss_2_1  0.0000


## 11. Phase-2 Completion Summary

Phase-2 artifacts are validated, leakage-free, and persisted. The imputed
splits in `data/splits/` are the certified input contract for Phase 3
(Demand Forecasting Engine): model selection will train on `train`, tune on
`val`, and report final metrics on `test` - never before.

In [11]:
pd.Series({
    "fact_rows_valid": len(fact),
    "products": matrix["product_id"].nunique(),
    "weeks": matrix["week"].nunique(),
    "matrix_rows": len(matrix),
    "train_rows": len(splits_imputed["train"]),
    "val_rows": len(splits_imputed["val"]),
    "test_rows": len(splits_imputed["test"]),
    "leakage_probe": "PASSED",
}), "Phase-2 pipeline completed."

(fact_rows_valid     112101
 products             32729
 weeks                  106
 matrix_rows        3469274
 train_rows         2428492
 val_rows            520391
 test_rows           520391
 leakage_probe       PASSED
 dtype: object,
 'Phase-2 pipeline completed.')